# Tutorial 03: BlueSky API
Author: Georg Ahnert

In this notebook, we will learn how to access data on the web through **API requests**.

This is different from _webscraping_, which we had a look at in the previous tutorials, since APIs provide data in a structured format.

However, APIs might not always be available or well-documented.

In this tutorial, we will focus on the [BlueSky (ATProto) API](https://docs.bsky.app/).

In [ ]:
# Install dependencies
!pip install requests pandas atproto tqdm ipywidgets

## REST API Basics

_APIs_ (Application Programming Interfaces) are aggreed upon standards that allows us (_clients_) to talk to a _server_ and request information from it, or provide it with information.
_REST_ (Representational State Transfer) describes a de-facto standard that many APIs on the web adhere to.

A central concept in REST APIs are _resources_. Each resource that we might want to interact with has a unique address, known as a _URI_ (Uniform Resource Identifier).

Different resources support different _HTTP methods_ or actions that you can perform on them.
The main HTTP methods are:

| HTTP method | action | description |
|---|---|---|
| GET	| Read | Retrieve data from the server |
| POST | Create | Send new data to the server to create a resource |
| PUT | Update | Replace an existing resource entirely |
| DELETE | Delete | Remove a resource from the server |

These methods are performed on dedicated _endpoints_ that support different types of interactions.
For example: https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile

You can directly try this out, simply by opening the endpoint above in your browser. By default, a GET request is performed.

We can also use a dedicated program like [Postman](https://www.postman.com/) to try out different types of requests. This makes it a bit easier to add parameters to a request or to use different HTTP methods.

Or you can use the requests package in Python. Let's try that out:

In [ ]:
import requests
import pandas as pd

In [3]:
url = "https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile"
response = requests.get(url) # if you remember from exercise 1, we had actually already used this to GET a webpage

The server response is now stored in the `response` variable and contains a lot of meta-information as well:

In [4]:
dict(response.headers)

{'Date': 'Tue, 03 Mar 2026 18:06:52 GMT',
 'Content-Type': 'application/json; charset=utf-8',
 'Content-Length': '85',
 'Connection': 'keep-alive',
 'Server': 'BunnyCDN-DE1-1330',
 'CDN-PullZone': '1816608',
 'CDN-RequestCountryCode': 'DE',
 'Access-Control-Allow-Origin': '*',
 'Cache-Control': 'public, max-age=5',
 'X-Powered-By': 'Express',
 'Strict-Transport-Security': 'max-age=63072000',
 'CDN-ProxyVer': '1.47',
 'CDN-RequestPullSuccess': 'True',
 'CDN-RequestPullCode': '400',
 'CDN-CachedAt': '03/03/2026 18:06:52',
 'CDN-EdgeStorageId': '1332',
 'CDN-RequestId': '9af31603395aefab9d9a207a2ec68861',
 'CDN-Cache': 'EXPIRED',
 'CDN-Status': '400',
 'CDN-RequestTime': '0'}

One of the most important pieces of information is the [HTTP status code](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes). It tells us if our request was successful or not.

The most common status codes are:

| Status Code | Status Text | Category | Description |
| :--- | :--- | :--- | :--- |
| **200** | OK | Success | The request was successful and the server sent data back. |
| **201** | Created | Success | The request was successful and a new resource was created. |
| **204** | No Content | Success | The request was successful, but there is no data to return. |
| **400** | Bad Request | Client Error | The server cannot process the request due to a client error. |
| **401** | Unauthorized | Client Error | Authentication is required and has failed or hasn't been provided. |
| **403** | Forbidden | Client Error | The server understood the request but refuses to authorize it. |
| **404** | Not Found | Client Error | The requested resource could not be found on the server. |
| **429** | Too Many Requests | Client Error | The user has sent too many requests in a given amount of time. |
| **500** | Internal Server Error | Server Error | A generic error message when the server encounters an unexpected condition. |
| **503** | Service Unavailable | Server Error | The server is currently unable to handle the request (overloaded/down). |

In [5]:
response.status_code # meh, Bad Request

400

Finally, let's have a look at the actual response that we got from the server on our GET request.

We can see that the server returned a JSON (JavaScript Object Notation) object that can be automatically converted into a Python dictionary.

In [6]:
response.text

'{"error":"InvalidRequest","message":"Error: Params must have the property \\"actor\\""}'

In [7]:
response.json()

{'error': 'InvalidRequest',
 'message': 'Error: Params must have the property "actor"'}

## Parameters in HTTP Requests

Okay, seems like we didn't use the BlueSky API properly and forgot to specify which profile we want to GET. Let's fix that:

In [8]:
url = "https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile"
params = {
    'actor': 'xkcd.com'
}

response = requests.get(url, params=params)
response.json()

{'did': 'did:plc:cz73r7iyiqn26upot4jtjdhk',
 'handle': 'xkcd.com',
 'displayName': 'Randall Munroe',
 'avatar': 'https://cdn.bsky.app/img/avatar/plain/did:plc:cz73r7iyiqn26upot4jtjdhk/bafkreibandvfr3i2qpwbuzfwcw26px2wfh2vkrgtbr2neiof4muzca2h2i@jpeg',
 'associated': {'lists': 0,
  'feedgens': 0,
  'starterPacks': 0,
  'labeler': False,
  'chat': {'allowIncoming': 'following'},
  'activitySubscription': {'allowSubscriptions': 'followers'}},
 'labels': [],
 'createdAt': '2023-04-27T20:54:28.631Z',
 'verification': {'verifications': [{'issuer': 'did:plc:z72i7hdynmk6r22z27h6tvur',
    'uri': 'at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky.graph.verification/3lndpuy2nfk2z',
    'isValid': True,
    'createdAt': '2025-04-21T10:47:23.196Z'}],
  'verifiedStatus': 'valid',
  'trustedVerifierStatus': 'none'},
 'indexedAt': '2024-01-20T05:04:56.709Z',
 'followersCount': 462977,
 'followsCount': 26,
 'postsCount': 581}

Maybe the most recent posts are a bit more interesting:

In [9]:
url = "https://public.api.bsky.app/xrpc/app.bsky.feed.getAuthorFeed"
params = {
    # see the documentation for possible parameters: https://docs.bsky.app/docs/api/app-bsky-feed-get-author-feed
    'actor': 'xkcd.com',
    'limit': 10,
}

response = requests.get(url, params=params)
response.json()

{'feed': [{'post': {'uri': 'at://did:plc:cz73r7iyiqn26upot4jtjdhk/app.bsky.feed.post/3mg46hoty222r',
    'cid': 'bafyreic4fc27vtnutxr4t6ditdjudjdkkmhoycozmqcqaoggjqeg6lj2z4',
    'author': {'did': 'did:plc:cz73r7iyiqn26upot4jtjdhk',
     'handle': 'xkcd.com',
     'displayName': 'Randall Munroe',
     'avatar': 'https://cdn.bsky.app/img/avatar/plain/did:plc:cz73r7iyiqn26upot4jtjdhk/bafkreibandvfr3i2qpwbuzfwcw26px2wfh2vkrgtbr2neiof4muzca2h2i@jpeg',
     'associated': {'chat': {'allowIncoming': 'following'},
      'activitySubscription': {'allowSubscriptions': 'followers'}},
     'labels': [],
     'createdAt': '2023-04-27T20:54:28.631Z',
     'verification': {'verifications': [{'issuer': 'did:plc:z72i7hdynmk6r22z27h6tvur',
        'uri': 'at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky.graph.verification/3lndpuy2nfk2z',
        'isValid': True,
        'createdAt': '2025-04-21T10:47:23.196Z'}],
      'verifiedStatus': 'valid',
      'trustedVerifierStatus': 'none'}},
    'record': {'$typ

In [10]:
# Inspecting the first post in the authorFeed a bit further
posts = response.json()['feed']
posts[0]

{'post': {'uri': 'at://did:plc:cz73r7iyiqn26upot4jtjdhk/app.bsky.feed.post/3mg46hoty222r',
  'cid': 'bafyreic4fc27vtnutxr4t6ditdjudjdkkmhoycozmqcqaoggjqeg6lj2z4',
  'author': {'did': 'did:plc:cz73r7iyiqn26upot4jtjdhk',
   'handle': 'xkcd.com',
   'displayName': 'Randall Munroe',
   'avatar': 'https://cdn.bsky.app/img/avatar/plain/did:plc:cz73r7iyiqn26upot4jtjdhk/bafkreibandvfr3i2qpwbuzfwcw26px2wfh2vkrgtbr2neiof4muzca2h2i@jpeg',
   'associated': {'chat': {'allowIncoming': 'following'},
    'activitySubscription': {'allowSubscriptions': 'followers'}},
   'labels': [],
   'createdAt': '2023-04-27T20:54:28.631Z',
   'verification': {'verifications': [{'issuer': 'did:plc:z72i7hdynmk6r22z27h6tvur',
      'uri': 'at://did:plc:z72i7hdynmk6r22z27h6tvur/app.bsky.graph.verification/3lndpuy2nfk2z',
      'isValid': True,
      'createdAt': '2025-04-21T10:47:23.196Z'}],
    'verifiedStatus': 'valid',
    'trustedVerifierStatus': 'none'}},
  'record': {'$type': 'app.bsky.feed.post',
   'createdAt': 

Based on the URI of this first post, we can investigate who liked it:

In [11]:
url = "https://public.api.bsky.app/xrpc/app.bsky.feed.getLikes"
params = {
    'uri': 'at://did:plc:cz73r7iyiqn26upot4jtjdhk/app.bsky.feed.post/3mg46hoty222r',
}

response = requests.get(url, params=params)
response.json()

{'likes': [{'actor': {'did': 'did:plc:wb3tevjffxyws4dh6mkoitc4',
    'handle': 'pulsargig.bsky.social',
    'displayName': '',
    'avatar': 'https://cdn.bsky.app/img/avatar/plain/did:plc:wb3tevjffxyws4dh6mkoitc4/bafkreigt342zzmmaq3hs4uuhrjlhd5n3fzrdniltklryya5p24ea2xwkge@jpeg',
    'associated': {'activitySubscription': {'allowSubscriptions': 'followers'}},
    'labels': [],
    'createdAt': '2024-11-22T03:53:35.853Z',
    'indexedAt': '2024-11-22T03:53:35.853Z'},
   'createdAt': '2026-03-03T18:05:51.153Z',
   'indexedAt': '2026-03-03T18:05:51.153Z'},
  {'actor': {'did': 'did:plc:fc3t6zpdthu7bgbrne3g4krg',
    'handle': 'sisyphianspartacus.bsky.social',
    'displayName': '',
    'avatar': 'https://cdn.bsky.app/img/avatar/plain/did:plc:fc3t6zpdthu7bgbrne3g4krg/bafkreic2u2tibbixdwuzlqxhtmijhec2kzh3jws32mm3zxsuitujredcme@jpeg',
    'associated': {'chat': {'allowIncoming': 'none'},
     'activitySubscription': {'allowSubscriptions': 'followers'}},
    'labels': [],
    'createdAt': '2025

In [12]:
actors = []
for like in response.json()['likes']:
    actors.append(like['actor'])

likers_df = pd.DataFrame(actors)

display(likers_df.head())
print(f'found {len(likers_df)} people who liked this post')

,did,handle,displayName,avatar,associated,labels,createdAt,indexedAt,description,pronouns
0,did:plc:wb3tevjffxyws4dh6mkoitc4,pulsargig.bsky.social,,https://cdn.bsky.app/img/avatar/plain/did:plc:...,{'activitySubscription': {'allowSubscriptions'...,[],2024-11-22T03:53:35.853Z,2024-11-22T03:53:35.853Z,NaN,NaN
1,did:plc:fc3t6zpdthu7bgbrne3g4krg,sisyphianspartacus.bsky.social,,https://cdn.bsky.app/img/avatar/plain/did:plc:...,"{'chat': {'allowIncoming': 'none'}, 'activityS...",[],2025-03-10T14:32:06.642Z,2025-05-04T22:32:03.275Z,NaN,NaN
2,did:plc:325f7xemyvavdxkkvbi67um6,1condor12.bsky.social,Steelperigren,NaN,{'activitySubscription': {'allowSubscriptions'...,[],2023-08-30T03:19:26.358Z,2024-01-20T06:38:12.379Z,He/him | 30 years old\nConsummate lurker,NaN
3,did:plc:4xn7om7w4ys7eorunku7n3w4,jjjennnnn.com,JJJENNNNN,https://cdn.bsky.app/img/avatar/plain/did:plc:...,{'activitySubscription': {'allowSubscriptions'...,[],2024-10-27T20:30:04.109Z,2025-05-10T20:15:16.540Z,🤔 - Jʼen doute.\n\n🇨🇦 in 🇺🇸,NaN
4,did:plc:usfbknt6ptd7nqaihbhdcikc,ouromakesgames.bsky.social,Ouroboros,https://cdn.bsky.app/img/avatar/plain/did:plc:...,{'activitySubscription': {'allowSubscriptions'...,[],2024-11-14T00:26:48.277Z,2026-01-31T23:12:12.346Z,What if we placed our Github repositories next...,NaN


found 50 people who liked this post


## Pageination

Notice how we only got 50 people as a response, while the post itself promised > 2000? That is due to a limited number of results that the BlueSky API returns by default.

We can increase this to 100 for a single request, but that is still much less than we would expect.

If we really want to capture all people who liked this post, we need to use _pageination_, i.e., do multiple requests using a _cursor_ and then combine the results.

In [13]:
url = "https://public.api.bsky.app/xrpc/app.bsky.feed.getLikes"
global_params = {
    'uri': 'at://did:plc:cz73r7iyiqn26upot4jtjdhk/app.bsky.feed.post/3mg46hoty222r',
    'limit': 100,
}
more_results = True
cursor = None
actors = []

# We continue our requests as long as there are more results
while more_results:
    # | allows us to add (multiple) new keys + values to an existing dict
    local_params = global_params | {
        'cursor': cursor
    }

    response = requests.get(url, params=local_params)
    json_response = response.json()

    # Save the actors that we found
    for like in json_response['likes']:
        actors.append(like['actor'])
    
    # A cursor is returned from the BlueSky API iff there are more results
    if 'cursor' in json_response:
        cursor = json_response['cursor']
    else:
        more_results = False
    
    # NOTE that you might want to include a small delay after each request.
    # Otherwise, you might run into rate limits and the server might reject your request.
    # We could also handle this by performing an exponential backoff when we encounter a status code 429.
    

all_likers_df = pd.DataFrame(actors)
all_likers_df

,did,handle,displayName,avatar,associated,labels,createdAt,indexedAt,description,pronouns,status,verification
0,did:plc:wb3tevjffxyws4dh6mkoitc4,pulsargig.bsky.social,,https://cdn.bsky.app/img/avatar/plain/did:plc:...,{'activitySubscription': {'allowSubscriptions'...,[],2024-11-22T03:53:35.853Z,2024-11-22T03:53:35.853Z,NaN,NaN,NaN,NaN
1,did:plc:fc3t6zpdthu7bgbrne3g4krg,sisyphianspartacus.bsky.social,,https://cdn.bsky.app/img/avatar/plain/did:plc:...,"{'chat': {'allowIncoming': 'none'}, 'activityS...",[],2025-03-10T14:32:06.642Z,2025-05-04T22:32:03.275Z,NaN,NaN,NaN,NaN
2,did:plc:325f7xemyvavdxkkvbi67um6,1condor12.bsky.social,Steelperigren,NaN,{'activitySubscription': {'allowSubscriptions'...,[],2023-08-30T03:19:26.358Z,2024-01-20T06:38:12.379Z,He/him | 30 years old\nConsummate lurker,NaN,NaN,NaN
3,did:plc:4xn7om7w4ys7eorunku7n3w4,jjjennnnn.com,JJJENNNNN,https://cdn.bsky.app/img/avatar/plain/did:plc:...,{'activitySubscription': {'allowSubscriptions'...,[],2024-10-27T20:30:04.109Z,2025-05-10T20:15:16.540Z,🤔 - Jʼen doute.\n\n🇨🇦 in 🇺🇸,NaN,NaN,NaN
4,did:plc:usfbknt6ptd7nqaihbhdcikc,ouromakesgames.bsky.social,Ouroboros,https://cdn.bsky.app/img/avatar/plain/did:plc:...,{'activitySubscription': {'allowSubscriptions'...,[],2024-11-14T00:26:48.277Z,2026-01-31T23:12:12.346Z,What if we placed our Github repositories next...,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2286,did:plc:o7e2hbtmvqexpbweuioy4ant,rsgreacen.bsky.social,Environmental Insurrectionist,https://cdn.bsky.app/img/avatar/plain/did:plc:...,"{'chat': {'allowIncoming': 'following'}, 'acti...",[],2024-04-19T10:52:08.698Z,2026-01-23T16:05:23.424Z,"Environmental advocate, coastal far Northern C...",NaN,NaN,NaN
2287,did:plc:56qd34vjsjb4y7f7p4fsgv6z,latinteachsmith.bsky.social,Bob Smith,https://cdn.bsky.app/img/avatar/plain/did:plc:...,"{'chat': {'allowIncoming': 'following'}, 'acti...","[{'src': 'did:plc:56qd34vjsjb4y7f7p4fsgv6z', '...",2024-04-20T11:46:51.442Z,2025-11-01T14:36:05.451Z,"citizen of planet Earth, Star Wars fan, board ...",NaN,NaN,NaN
2288,did:plc:g6zf2sowg6z7h6kbsrzwtzuf,arcticfoxnetwork.bsky.social,No Name (It/We)🏳️‍⚧️,https://cdn.bsky.app/img/avatar/plain/did:plc:...,{'activitySubscription': {'allowSubscriptions'...,[],2025-12-27T10:12:20.424Z,2026-01-29T17:04:21.546Z,NaN,NaN,NaN,NaN
2289,did:plc:tvzetckkl7vpvzm7sdsg6xop,phroodloops.bsky.social,Pink Freud,https://cdn.bsky.app/img/avatar/plain/did:plc:...,{'activitySubscription': {'allowSubscriptions'...,[],2024-12-04T19:04:56.445Z,2025-11-25T01:18:03.224Z,"""Insomnio tiene el que no está durmiendo con e...",NaN,NaN,NaN


## ATProto Python Client

For ATProto, the protocol behind the BlueSky API, there is actually a python wrapper that lets us gather this kind of data a bit more conveniently.

It allows us to use Python objects directly and it also handles rate limits.

In [ ]:
from atproto import Client

In [15]:
client = Client(base_url='https://public.api.bsky.app') # use the public API without authentification
client

In [16]:
response = client.get_likes('at://did:plc:cz73r7iyiqn26upot4jtjdhk/app.bsky.feed.post/3mg46hoty222r')
response.likes

[Like(actor=ProfileView(did='did:plc:wb3tevjffxyws4dh6mkoitc4', handle='pulsargig.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=None, labeler=None, lists=None, starter_packs=None, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:wb3tevjffxyws4dh6mkoitc4/bafkreigt342zzmmaq3hs4uuhrjlhd5n3fzrdniltklryya5p24ea2xwkge@jpeg', created_at='2024-11-22T03:53:35.853Z', debug=None, description=None, display_name='', indexed_at='2024-11-22T03:53:35.853Z', labels=[], pronouns=None, status=None, verification=None, viewer=None, py_type='app.bsky.actor.defs#profileView'), created_at='2026-03-03T18:05:51.153Z', indexed_at='2026-03-03T18:05:51.153Z', py_type='app.bsky.feed.getLikes#like'),
 Like(actor=ProfileView(did='did:plc:fc3t6zpdthu7bgbrne3g4krg', handle='sisyphianspart

In [17]:
pd.DataFrame([dict(liker['actor']) for liker in response.likes]).head()

,did,handle,associated,avatar,created_at,debug,description,display_name,indexed_at,labels,pronouns,status,verification,viewer,py_type
0,did:plc:wb3tevjffxyws4dh6mkoitc4,pulsargig.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-11-22T03:53:35.853Z,None,NaN,,2024-11-22T03:53:35.853Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
1,did:plc:fc3t6zpdthu7bgbrne3g4krg,sisyphianspartacus.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2025-03-10T14:32:06.642Z,None,NaN,,2025-05-04T22:32:03.275Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
2,did:plc:325f7xemyvavdxkkvbi67um6,1condor12.bsky.social,activity_subscription=ProfileAssociatedActivit...,NaN,2023-08-30T03:19:26.358Z,None,He/him | 30 years old\nConsummate lurker,Steelperigren,2024-01-20T06:38:12.379Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
3,did:plc:4xn7om7w4ys7eorunku7n3w4,jjjennnnn.com,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-10-27T20:30:04.109Z,None,🤔 - Jʼen doute.\n\n🇨🇦 in 🇺🇸,JJJENNNNN,2025-05-10T20:15:16.540Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
4,did:plc:usfbknt6ptd7nqaihbhdcikc,ouromakesgames.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-11-14T00:26:48.277Z,None,What if we placed our Github repositories next...,Ouroboros,2026-01-31T23:12:12.346Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView


Again, let's add pageination:

In [18]:
post_uri = 'at://did:plc:cz73r7iyiqn26upot4jtjdhk/app.bsky.feed.post/3mg46hoty222r'

more_results = True
cursor = None
actors = []

# We continue our requests as long as there are more results
while more_results:
    response = client.get_likes(post_uri, cursor=cursor, limit=100)
    actors += [dict(liker['actor']) for liker in response.likes]

    # A cursor is returned from the BlueSky API iff there are more results
    if response.cursor is not None:
        cursor = response.cursor
    else:
        more_results = False

likers_df2 = pd.DataFrame(actors)
likers_df2

,did,handle,associated,avatar,created_at,debug,description,display_name,indexed_at,labels,pronouns,status,verification,viewer,py_type
0,did:plc:wb3tevjffxyws4dh6mkoitc4,pulsargig.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-11-22T03:53:35.853Z,None,NaN,,2024-11-22T03:53:35.853Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
1,did:plc:fc3t6zpdthu7bgbrne3g4krg,sisyphianspartacus.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2025-03-10T14:32:06.642Z,None,NaN,,2025-05-04T22:32:03.275Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
2,did:plc:325f7xemyvavdxkkvbi67um6,1condor12.bsky.social,activity_subscription=ProfileAssociatedActivit...,NaN,2023-08-30T03:19:26.358Z,None,He/him | 30 years old\nConsummate lurker,Steelperigren,2024-01-20T06:38:12.379Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
3,did:plc:4xn7om7w4ys7eorunku7n3w4,jjjennnnn.com,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-10-27T20:30:04.109Z,None,🤔 - Jʼen doute.\n\n🇨🇦 in 🇺🇸,JJJENNNNN,2025-05-10T20:15:16.540Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
4,did:plc:usfbknt6ptd7nqaihbhdcikc,ouromakesgames.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-11-14T00:26:48.277Z,None,What if we placed our Github repositories next...,Ouroboros,2026-01-31T23:12:12.346Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2286,did:plc:o7e2hbtmvqexpbweuioy4ant,rsgreacen.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-04-19T10:52:08.698Z,None,"Environmental advocate, coastal far Northern C...",Environmental Insurrectionist,2026-01-23T16:05:23.424Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
2287,did:plc:56qd34vjsjb4y7f7p4fsgv6z,latinteachsmith.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-04-20T11:46:51.442Z,None,"citizen of planet Earth, Star Wars fan, board ...",Bob Smith,2025-11-01T14:36:05.451Z,[cts='2024-04-20T11:46:53.048Z' src='did:plc:5...,NaN,None,None,None,app.bsky.actor.defs#profileView
2288,did:plc:g6zf2sowg6z7h6kbsrzwtzuf,arcticfoxnetwork.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2025-12-27T10:12:20.424Z,None,NaN,No Name (It/We)🏳️‍⚧️,2026-01-29T17:04:21.546Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView
2289,did:plc:tvzetckkl7vpvzm7sdsg6xop,phroodloops.bsky.social,activity_subscription=ProfileAssociatedActivit...,https://cdn.bsky.app/img/avatar/plain/did:plc:...,2024-12-04T19:04:56.445Z,None,"""Insomnio tiene el que no está durmiendo con e...",Pink Freud,2025-11-25T01:18:03.224Z,[],NaN,None,None,None,app.bsky.actor.defs#profileView


## Authentification

The _public_ BlueSky API already allows you to do a lot of things without needing to login.
However, for specific actions, a login is required.

You can set up an app password that can be revoked easier and does not give your Python script access to your private messages: https://bsky.app/settings/privacy-and-security

While BlueSky already allows you to do many things with the public API, other services might not have a public API or might impose much stricter rate limits without login.
For instance, when using LLMs through the APIs of some of the common providers (OpenAI, Google, Anthropic, etc.), you have to login so that your LLM usage can be tracked and billed accordingly.
Make sure to check the documentation for each API that you might want to use.

In [ ]:
pwd_client = Client()
pwd_client.login(
    login = 'wanlo.bsky.social',
    password = '' # TODO: add you own app password
)

In [50]:
feed = pwd_client.get_author_feed('netzpolitik.org')
latest_post = feed.feed[0].post
latest_post

PostView(author=ProfileViewBasic(did='did:plc:bd2ad25frux6jav66chr6aiw', handle='netzpolitik.org', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=ProfileAssociatedChat(allow_incoming='following', py_type='app.bsky.actor.defs#profileAssociatedChat'), feedgens=None, labeler=None, lists=None, starter_packs=None, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:bd2ad25frux6jav66chr6aiw/bafkreiad2h326vig3y2ruamzyt447eh2wwnwgda7k33himl4utdeftzgqy@jpeg', created_at='2023-09-27T13:49:36.871Z', debug=None, display_name='netzpolitik.org', labels=[], pronouns=None, status=None, verification=VerificationState(trusted_verifier_status='none', verifications=[VerificationView(created_at='2025-07-24T01:40:45.708Z', is_valid=True, issuer='did:plc:z72i7hdynmk6r22z27h6tvur', uri='at://did:plc:z72i7hdynmk

In [51]:
pwd_client.app.bsky.bookmark.create_bookmark(
    {
        "uri": latest_post.uri,
        # CIDs go beyond URIs and allow you to reference a specific version of a post
        "cid": latest_post.cid,
    }
)

True

## Task: Identify BlueSky accounts that have recently reposted @netzpolitik.org

Based on the display names and self-descriptions of these accounts, would you say that there are many bots?

In [ ]:
# Import tqdm for a progress bar
from tqdm.auto import tqdm

In [ ]:
# Retrieve the first 50 posts
feed = client.get_author_feed('netzpolitik.org')

In [ ]:
actors = []

# Iterate over the first 50 posts that we retrieved
for feed_post in tqdm(feed.feed):
    # skip reposts by netzpolitik.org
    if feed_post.post.author.handle != 'netzpolitik.org':
        continue

    more_results = True
    cursor = None

    # We continue our requests as long as there are more results
    while more_results:
        # Get all actors that reposted this post
        response = client.get_reposted_by(feed_post.post.uri, cursor=cursor, limit=100)
        for reposter in response.reposted_by:
            actors.append({
                'handle': reposter.handle,
                'display_name': reposter.display_name,
                'description': reposter.description,
                'post_uri': feed_post.post.uri,
            })

        # A cursor is returned from the BlueSky API iff there are more results
        if response.cursor is not None:
            cursor = response.cursor
        else:
            more_results = False

reposters_df = pd.DataFrame(actors)
reposters_df

100%|██████████| 50/50 [00:08<00:00,  5.84it/s]


,display_name,description,post_uri
0,Clarence,NaN,at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
1,Dagmar Hoffmann,Prof of Media and Communication | Gender Media...,at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
2,No Idea But Science,"Science, SciFi, IT, F(L)OSS, Covid (not over) ...",at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
3,Erdmeister,Am Abend fühle ich mich dem Morgen gewachsen.\n,at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
4,Eudoxus,#FCKNZS #FCKAFD #AfDVerbot 333ppm,at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
...,...,...,...
1723,,NaN,at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
1724,Roger Raeder°,| Musikfreak | Lehrer FöL PoWi | Christ | Drum...,at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
1725,Mutter des Chaos,Mutter des großen (K1) und des kleinen Chaos (K2),at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
1726,Simon,"Radsport, Menschenrechte, Arbeitnehmer*Innenre...",at://did:plc:bd2ad25frux6jav66chr6aiw/app.bsky...
